# NHCS Scenario 3 — Not sent home: who leaves the emergency department for somewhere else

_A model of the visit outcome on 1.3 million records. ED dataset only. Level: second analysis, first model. About 90 minutes._

**Before you start.** The NHCS pathology holds two datasets from about 200 US hospitals, 2020 to
2022: 1,368,648 emergency department visit records and 488,636 inpatient discharge records, each a
5% sample of all encounters of the participating hospitals. **A row is an encounter record, not a
patient**, and the two datasets cannot be linked: each is an independent sample, the identifiers
were renumbered, and NCHS merged the ED outcome "admitted" into "other". The MIP does not use the
survey weights: every figure describes *the sampled encounters*, not US hospital care. Codes are
NCHS codes: Sex 1 = Male, 2 = Female; NCHS flags 1 = Yes, 2 = No; curated flags 0 = No, 1 = Yes;
age is top-coded at 86 and the length of stay at 15 days.

**How this notebook works.** Every analysis is a *federated* call: the code selects a dataset and
variables, defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a
model). No record ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect. These are the largest datasets of this MIP: the notebook prints how
long the regressions take.

**Learning goals.** Work with an outcome whose most important category was masked; relate an
outcome to age, comorbidity, acute severity and the reason for the visit with cross-tabulations;
compare two groups with a t-test built from descriptive statistics; fit a logistic regression on
more than a million rows and read its odds ratios; and measure how long the platform takes.

**Clinical background.** For an emergency physician the central decision is disposition: home, or
not home. "Not home" means admission to a bed, transfer to another hospital or a hospice, home with
a home health service, death in the ED, or the patient leaving against advice. NCHS merged
"admitted" with "other" before release, so the dataset cannot say who was admitted, but it can say
who went home and who did not. The candidates that drive that decision are in every record: age,
the Charlson comorbidity index (a weighted count of 17 chronic conditions, 0 for most ED patients),
sepsis and COVID-19 among the diagnoses, and whether the visit was for an injury.

## 1. Connect to the MIP and open the NHCS pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("NHCS")        # add version="ED_Inpatient_v1" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the platform labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

ed = dataset("nhcs_ed", "NHCS emergency department visits, USA, 2020-2022")
ip = dataset("nhcs_ip", "NHCS inpatient discharges, USA, 2020-2022")
display(dm)
display(dm.datasets.to_frame())

In [ ]:
# The variable tree is the anatomy of one encounter record: Encounter, Patient, Discharge,
# Diagnoses (with the curated condition flags and the NCHS CCSR flags), and the branch
# Inpatient stay (empty for ED visits): length of stay and procedures.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the selected datasets
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False, datasets=None):
    """A Pipeline on the datasets of this notebook (DATASETS, or the ones given) with the given
    variables and cohort filter. Tests and models need drop_missing=True: the platform requires an
    explicit missing-values step and analyses complete cases only. describe() keeps it off so it
    can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable, dataset=POOLED):
    """The describe row of one variable: the pooled row by default, or the row of one Dataset."""
    keys = (POOLED,) if dataset == POOLED else (dataset._code, dataset.label)
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") in keys]


def numeric_summary(result, variables, dataset=POOLED):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v, dataset):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable, dataset=POOLED):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable, dataset):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan, "missing": (r.get("data") or {}).get("num_na")})
    return pd.DataFrame(rows)


def level_percent(result, variable, levels=("Yes",), dataset=POOLED):
    """Percentage of the given levels (labels) of a categorical variable in a describe result."""
    k = category_counts(result, variable, dataset)
    return float(k.loc[k["level"].isin(levels), "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None, datasets=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def band_means(numeric, bands, base_filter=None, datasets=None):
    """n, mean and median of a numerical variable in bands given as {label: filter}."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({"band": label, "n": d.get("num_dtps"), "mean": d.get("mean"), "median": d.get("q2")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None, event_label=None, datasets=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort.
    `event` is the outcome code counted in the percentage column, or a list of codes."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    events = [event] if isinstance(event, str) else list(event)
    pct_col = "% " + (event_label or lab_y.get(events[0], events[0]))
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f, datasets=datasets).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row[pct_col] = 100 * sum(counts.get(e, 0) for e in events) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True, datasets=datasets).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def rates_by(group, outcomes, levels=None, base_filter=None, event="1", datasets=None):
    """% of records with each outcome (code `event`) in every level of `group`: one describe per
    level for the counts of all outcomes, one Chi-squared Test per outcome (in .attrs)."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f, datasets=datasets).describe()
        row = {group.label: lab.get(c, c)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.attrs["chi2"], table.attrs["p"] = {}, {}
    for y in outcomes:
        f = F(group).is_not_null() & F(y).is_not_null()
        if levels:
            f = f & F(group).isin(list(levels))
        if base_filter is not None:
            f = f & base_filter
        chi = analysis([group, y], f, drop_missing=True, datasets=datasets).chi_square_test(x=group, y=y)
        table.attrs["chi2"][y.label] = chi.raw.get("chi2")
        table.attrs["p"][y.label] = chi.raw.get("p_value")
    return table


def month_label(i):
    """Month index 1..36 -> '2020-01' .. '2022-12'."""
    return f"{2020 + (i - 1) // 12}-{(i - 1) % 12 + 1:02d}"


def by_month(outcomes, variable, months=range(1, 37), datasets=None):
    """% of records with each outcome (code 1) per month of the time axis: one describe per month."""
    rows = []
    for m in months:
        res = analysis(outcomes, F(variable) == m, datasets=datasets).describe()
        row = {"month": m, "label": month_label(m)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get("1", 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None, rotate=0):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: PCT}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:,.0f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(7, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=rotate)
    plt.tight_layout()
    plt.show()


def anova_summary(result):
    """One-way ANOVA result -> F, p, n, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), table.get("n_obs"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def n_fmt(value):
    # a count with thousands separators, or n/a when the platform did not return it
    return f"{value:,}" if isinstance(value, (int, float)) else "n/a"


def ttest_from_describe(result_a, result_b, variable):
    """The pooled-variance t statistic of two groups from their describe results (n, mean, SD):
    all a t-test needs, for groups the platform's t-test cannot form (several codes pooled)."""
    a = numeric_summary(result_a, [variable]).iloc[0]
    b = numeric_summary(result_b, [variable]).iloc[0]
    n1, n2 = a["n"], b["n"]
    sp2 = ((n1 - 1) * a["sd"] ** 2 + (n2 - 1) * b["sd"] ** 2) / (n1 + n2 - 2)
    t = (a["mean"] - b["mean"]) / math.sqrt(sp2 * (1 / n1 + 1 / n2))
    return {"mean_a": a["mean"], "mean_b": b["mean"], "diff": a["mean"] - b["mean"], "t": t, "n": n1 + n2}


def term_label(term, variables):
    """Model term -> readable label: 'SEX[2]' -> 'Sex = Female', 'AGE' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(lambda t: term_label(t, variables))
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def linear_table(result, variables):
    """Linear regression result -> one row per term with coefficient (units of the outcome), 95% CI and p."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return pd.DataFrame({"term": [term_label(t, variables) for t in raw.get("indep_vars", [])],
                         "coefficient": raw.get("coefficients"), "ci_low": raw.get("lower_ci"),
                         "ci_high": raw.get("upper_ci"), "p": raw.get("pvalues")})


def show_coefficients(table, title, unit="days"):
    """Table and dot chart of linear-regression coefficients (the intercept left out of the chart)."""
    display(table.style.format({"coefficient": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table[table["term"] != "Intercept"].iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["coefficient"], range(len(t)),
                xerr=[t["coefficient"] - t["ci_low"], t["ci_high"] - t["coefficient"]], fmt="o", capsize=4)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel(f"{unit} added, for the same values of the other predictors")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The dataset and the variables of this scenario

| Code | Label in the MIP | Tree location (under NHCS) | Role | Values |
|---|---|---|---|---|
| `discharge_group` | Discharge group | Discharge | outcome: sent home (1) or not; curation-derived | 1 Home, routine; 2 Home with home health care; 3 Transfer to another facility or hospice; 4 Left against medical advice; 5 Died; 6 Other (for ED visits includes admission) |
| `age_group` | Age group | Patient | predictor, grouped; curation-derived | 1 Under 1 year; 2 1-17 years; 3 18-44 years; 4 45-64 years; 5 65-74 years; 6 75-84 years; 7 85 years and over |
| `AGE` | Age | Patient | predictor, continuous | integer, 0-86 years |
| `SEX` | Sex | Patient | predictor | 1 Male; 2 Female |
| `charlson_group` | Charlson index group | Diagnoses / Condition flags (curated) | predictor: comorbidity; curation-derived | 0 0 (no Charlson condition); 1 1-2; 2 3-4; 3 5 or more |
| `n_chronic` | Number of chronic conditions | Diagnoses / Condition flags (curated) | chronic conditions, as a mean; curation-derived | integer, 0-15 |
| `sepsis_dx` | Sepsis, any position | Diagnoses / Condition flags (curated) | predictor: acute severity; curation-derived | 0 No; 1 Yes |
| `covid_dx` | COVID-19, any position | Diagnoses / Condition flags (curated) | predictor; curation-derived | 0 No; 1 Yes |
| `cond_injury` | Injury diagnosis | Diagnoses / Condition flags (curated) | predictor: reason for the visit; curation-derived | 0 No; 1 Yes |
| `YEAR` | Survey year | Encounter | extension | 2020 2020; 2021 2021; 2022 2022 |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `AGE`) is what the csv
column is called; you will see it in some result payloads. The first line below chooses the
dataset(s) every call of this notebook uses.

In [ ]:
DATASETS = [ed]                          # every call of this notebook runs on the ED visits

outcome    = dm.variables["Discharge group"]
age_group  = dm.variables["Age group"]
age        = dm.variables["Age"]
sex        = dm.variables["Sex"]
charlson   = dm.variables["Charlson index group"]
n_chronic  = dm.variables["Number of chronic conditions"]
sepsis     = dm.variables["Sepsis, any position"]
covid      = dm.variables["COVID-19, any position"]
injury     = dm.variables["Injury diagnosis"]
year       = dm.variables["Survey year"]
display(mip.to_frame([outcome, age_group, age, sex, charlson, n_chronic, sepsis, covid, injury, year]))

## Step 1. How visits end

*Expected:* home, routine 1,136,470 (87.3% of the 1,302,296 known outcomes); home with home
health care 21,180 (1.6%); transfer to another facility or hospice 27,782 (2.1%); left against
medical advice 40,703 (3.1%); died 6,796 (0.5%); other, which includes admission, 69,365 (5.3%);
66,352 visits without a recorded outcome.

In [ ]:
ended = category_counts(analysis([outcome]).describe(), outcome)
display(ended[["level", "count", "percent", "missing"]].style.format({"percent": PCT}).hide(axis="index"))

## Step 2. By age

*Expected, sent home:* 96.1% under 1 year, 95.8% at 1 to 17, 91.3% at 18 to 44, 86.6% at 45 to
64, 78.6% at 65 to 74, 71.4% at 75 to 84, 61.9% at 85 and over (chi-square 119,782, p < 0.001).
The "other" share rises from 1.3% to 18.8% along the same groups.

In [ ]:
by_age = crosstab(age_group, outcome, event="1", event_label="sent home")
other = labels_of(outcome)["6"]
by_age["% other (includes admission)"] = 100 * by_age[other] / by_age["n"]
display(by_age[[age_group.label, "n", "% sent home", "% other (includes admission)"]].style.format({"% sent home": PCT, "% other (includes admission)": PCT}).hide(axis="index"))
print(f"Chi-squared Test: chi2 = {by_age.attrs['chi2']:,.0f}, p = {fmt_p(by_age.attrs['p'])}")
ax = by_age.set_index(age_group.label)[["% sent home", "% other (includes admission)"]].plot(kind="bar", figsize=(8, 3.4), rot=30, color=["#4C72B0", "#C44E52"])
ax.set_ylabel("% of the visits of the age group"); ax.set_xlabel(""); ax.set_title("Sent home, and 'other', by age group")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Step 3. Comorbidity, severity and the reason for the visit

The share sent home by Charlson group, sepsis, COVID-19, injury and sex: one cross-tabulation per
factor.

*Expected, sent home:* Charlson 0: 90.5%, 1-2: 81.7%, 3-4: 64.9%, 5 or more: 60.4%; sepsis no
88.1%, yes 31.7%; COVID-19 no 87.4%, yes 84.0%; injury no 86.6%, yes 90.5%; men 86.3%, women
88.1%. All p < 0.001 (the chi-square of the six-level discharge group by the Charlson group is
102,085). Comorbidity halves the chance of going home across its range; sepsis is the strongest
single reason not to; an injury visit goes home more often than an illness visit; COVID-19, in
the ED, changes little.

In [ ]:
factors = []
for exposure in (charlson, sepsis, covid, injury, sex):
    t = crosstab(exposure, outcome, event="1", event_label="sent home")
    print(f"{exposure.label}: chi2 = {t.attrs['chi2']:,.0f}, p = {fmt_p(t.attrs['p'])}")
    for _, r in t.iterrows():
        factors.append({"factor": exposure.label, "level": r[exposure.label], "n": r["n"], "% sent home": r["% sent home"]})
factors = pd.DataFrame(factors)
display(factors.style.format({"% sent home": PCT}).hide(axis="index"))
fig, ax = plt.subplots(figsize=(8, 4.5))
labels = factors["factor"] + ": " + factors["level"].astype(str)
ax.barh(labels[::-1], factors["% sent home"][::-1], color="#4C72B0")
ax.set_xlabel("% sent home"); ax.set_title("Sent home, by each factor")
plt.tight_layout()
plt.show()

## Step 4. Age and chronic conditions, as means

The platform's t-test needs two codes of one variable; the second group here pools five codes, so
the notebook takes one `describe()` per group and computes the t statistic from the two means,
standard deviations and counts, which is all a t-test needs.

*Expected:* mean age 39.3 years when sent home against 55.9 when not (t about 270); 0.77 chronic
conditions against 1.44 (t about 190).

In [ ]:
home = analysis([age, n_chronic], F(outcome) == "1").describe()
not_home = analysis([age, n_chronic], (F(outcome) != "1") & F(outcome).is_not_null()).describe()
for v in (age, n_chronic):
    s = ttest_from_describe(home, not_home, v)
    print(f"{v.label}: sent home {s['mean_a']:.2f}, not sent home {s['mean_b']:.2f}, difference {s['diff']:.2f}, t = {s['t']:.0f}, n = {s['n']:,}")

## Step 5. The model: who goes home

A *Logistic Regression* of the discharge group with the positive class 1 = home, routine (the
platform turns the five other codes into "not home") on age, sex, Charlson group, COVID-19,
sepsis and injury. The largest regression of the teaching material: the cell times it.

*Expected (1,299,560 visits with a recorded outcome, age, sex and condition flags), odds of going
home:* age 0.975 per year (0.78 per 10 years); women
1.17 (1.16 to 1.18); Charlson 1-2 0.76, 3-4 0.42, 5 or more 0.34 (versus 0); COVID-19 0.97 (0.95
to 1.00); sepsis 0.11 (0.11 to 0.12); injury 1.30 (1.28 to 1.32). Every factor keeps its
direction after adjustment. Sepsis divides the odds of going home by nine, a Charlson index of 5
or more by three, each decade of age by 1.3. The confidence intervals are narrow because the
sample is huge, which says nothing about whether the estimates are unbiased.

In [ ]:
covariates = [age, sex, charlson, covid, sepsis, injury]
t0 = time.perf_counter()
model = analysis([outcome] + covariates, drop_missing=True).logistic_regression(x=covariates, y=outcome, positive_class="1")
print(f"{model.raw['summary']['n_obs']:,} visits in the model; the platform answered in {time.perf_counter() - t0:.0f} s")
ors = odds_ratio_table(model, covariates)
show_odds_ratios(ors, "Odds of being sent home, sampled ED visits 2020 to 2022")
per_year = ors.set_index("term").loc["Age (per unit)"]
print(f"Age, per 10 years: {per_year['odds_ratio'] ** 10:.2f}")

## Step 6. State the result

A model answer:

> Among 1.3 million sampled US emergency department visits of 2020 to 2022, 87% ended with the
> patient going home, from 96% under 18 years to 62% at 85 and over. Going home was less likely
> with age (odds ratio 0.78 per 10 years), with comorbidity (0.34 for a Charlson index of 5 or
> more) and above all with sepsis (0.11), and more likely after an injury (1.30) and for women
> (1.17); COVID-19 made little difference (0.97). Because NCHS merged admission into "other",
> these figures describe the chance of going home, not the chance of admission; they are
> unweighted sample estimates.

**Questions for discussion.** "Not home" mixes admission, transfer, home care, death and leaving
against advice: for which factors does that mixture matter most? The odds ratio for COVID-19 is
0.97 in the ED while Scenario 4 finds 3.2 for death in hospital: how can both be true? What would
the confidence interval for age be with 13,000 visits instead of 1.3 million? How long did the
regression take, and what does that say about using the MIP on a national dataset?

**Optional extension.** Add the survey year to the model: did the chance of going home change over
the three years, for the same kind of patient?

In [ ]:
covariates = [age, sex, charlson, covid, sepsis, injury, year]
model = analysis([outcome] + covariates, drop_missing=True).logistic_regression(x=covariates, y=outcome, positive_class="1")
display(odds_ratio_table(model, covariates).style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": fmt_p}).hide(axis="index"))